<a href="https://colab.research.google.com/github/taselshambakey/DECI-final-project/blob/main/Tasneem_DECI_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sqlite3

conn = sqlite3.connect('database.db')
conn.row_factory = sqlite3.Row
cur = conn.cursor()

# Dedup view: 8 checkout_ids appear as exact duplicate rows in the raw table.
# We collapse them so no checkout is double-counted.
cur.execute("DROP VIEW IF EXISTS checkouts_clean")
cur.execute("""
    CREATE VIEW checkouts_clean AS
    SELECT DISTINCT checkout_id, member_id, book_id, checkout_date, return_date
    FROM checkouts
""")

output_lines = []

def log(line=""):
    print(line)
    output_lines.append(str(line))

log("################ Q1: Checkouts per member (incl. zero) ################")
q1 = """
SELECT m.member_id,
       m.first_name || ' ' || m.last_name AS member_name,
       COUNT(c.checkout_id) AS checkout_count
FROM members m
LEFT JOIN checkouts_clean c ON c.member_id = m.member_id
GROUP BY m.member_id, member_name
ORDER BY checkout_count DESC, m.member_id ASC;
"""
cur.execute(q1)
rows1 = cur.fetchall()
for r in rows1:
    log(dict(r))
log(f"Total members: {len(rows1)}")
log(f"Members with 0 checkouts: {sum(1 for r in rows1 if r['checkout_count'] == 0)}")

log()
log("################ Q2: Author pattern search ################")
# Chosen pattern: authors whose first name starts with 'A'
pattern = 'A%'
q2 = """
SELECT book_id, title, author
FROM books
WHERE author LIKE ?
ORDER BY author, title;
"""
cur.execute(q2, (pattern,))
rows2 = cur.fetchall()
for r in rows2:
    log(dict(r))

log()
log("################ Q3: Top 5 most-borrowed titles ################")
q3 = """
SELECT b.book_id, b.title, b.author, COUNT(c.checkout_id) AS times_borrowed
FROM checkouts_clean c
JOIN books b ON b.book_id = c.book_id
GROUP BY b.book_id, b.title, b.author
ORDER BY times_borrowed DESC, b.title ASC
LIMIT 5;
"""
cur.execute(q3)
rows3 = cur.fetchall()
for r in rows3:
    log(dict(r))

log()
log("################ Q4: Top 10 most active readers ################")
q4 = """
SELECT m.member_id,
       m.first_name || ' ' || m.last_name AS member_name,
       COUNT(c.checkout_id) AS checkout_count
FROM members m
JOIN checkouts_clean c ON c.member_id = m.member_id
GROUP BY m.member_id, member_name
ORDER BY checkout_count DESC, m.member_id ASC
LIMIT 10;
"""
cur.execute(q4)
rows4 = cur.fetchall()
for r in rows4:
    log(dict(r))

log()
log("################ Q5: Neighborhood activity, skipping 10 most recent ################")
# Chosen neighborhood: Maadi (normalized for casing/whitespace variants)
neighborhood = 'maadi'
q5 = """
SELECT c.checkout_id, m.member_id,
       m.first_name || ' ' || m.last_name AS member_name,
       c.book_id, c.checkout_date, c.return_date
FROM checkouts_clean c
JOIN members m ON m.member_id = c.member_id
WHERE LOWER(TRIM(m.neighborhood)) = ?
ORDER BY c.checkout_date DESC, c.checkout_id DESC
LIMIT -1 OFFSET 10;
"""
cur.execute(q5, (neighborhood,))
rows5 = cur.fetchall()
for r in rows5:
    log(dict(r))
log(f"Total Maadi checkouts: {len(rows5) + 10}  | shown (beyond most recent 10): {len(rows5)}")

conn.close()

# Save all answers to a plain text file
output_path = "answers.txt"
with open(output_path, "w") as f:
    f.write("\n".join(output_lines))

print(f"\nSaved answers to {output_path}")

################ Q1: Checkouts per member (incl. zero) ################
{'member_id': 1034, 'member_name': 'Aya Wahba', 'checkout_count': 23}
{'member_id': 1044, 'member_name': 'Sherif Saleh', 'checkout_count': 20}
{'member_id': 1008, 'member_name': 'Ziad Saleh', 'checkout_count': 19}
{'member_id': 1010, 'member_name': 'Nour Nabil', 'checkout_count': 18}
{'member_id': 1027, 'member_name': 'Mostafa Fouad', 'checkout_count': 18}
{'member_id': 1018, 'member_name': 'Ahmed Shafik', 'checkout_count': 17}
{'member_id': 1024, 'member_name': 'Youssef Hegazy', 'checkout_count': 16}
{'member_id': 1030, 'member_name': 'Reem Osman', 'checkout_count': 16}
{'member_id': 1047, 'member_name': 'Sara Rashad', 'checkout_count': 16}
{'member_id': 1065, 'member_name': 'Adam Fahmy', 'checkout_count': 15}
{'member_id': 1072, 'member_name': 'Seif Zaki', 'checkout_count': 14}
{'member_id': 1050, 'member_name': 'Fares Adel', 'checkout_count': 11}
{'member_id': 1057, 'member_name': 'Ahmed Fahmy', 'checkout_count'